# GATv2 heterogéneo corregido — TEA vs. HC

Versión completa para Kaggle/Jupyter. Mantiene **paciente ↔ bacteria**, **bacteria ↔ bacteria (TEA)** y **bacteria ↔ bacteria (HC)**. La estructura de los tres tipos de aristas es no dirigida.

Cambios principales: embeddings de taxón; rama directa de abundancia + presencia; compuerta residual ecológica; grafo ecológico construido solo con entrenamiento; *early stopping* interno; reentrenamiento por fold externo; predicciones fuera de muestra (*OOF*); control directo frente a red heterogénea.

**Importante:** alcanzar 90% no está garantizado; los resultados del test externo no deben utilizarse para seleccionar hiperparámetros.

## 1. Configuración, carga del dataset y construcción de los grafos

La ruta del CSV se detecta automáticamente. También podés configurar `CFG.data_path`. Para probar un valor distinto de `top_k` o `corr_threshold`, modificalo en `CFG`; ambas opciones se usan efectivamente en la construcción de redes.

In [1]:
# -*- coding: utf-8 -*-
"""
GAT heterogeneo para microbiota intestinal: HC (0) vs TEA (1).

Grafo de estructura NO DIRIGIDA:
    paciente <-> bacteria (abundancia del paciente)
    bacteria <-> bacteria, red TEA (correlaciones de TRAIN)
    bacteria <-> bacteria, red HC  (correlaciones de TRAIN)

Incluye:
 - Identidad de bacterias aprendible + ocho atributos ecologicos.
 - Rama de abundancias/presencia directa y correccion residual ecologica.
 - Grafos por fold sin usar las etiquetas del conjunto de prueba.
 - Validacion cruzada externa de 5 folds y validacion interna para elegir epocas.
 - Reentrenamiento con TODO el train externo antes de la evaluacion externa.
 - Comparacion directo / bipartito / heterogeneo con las mismas particiones.
 - CSV de predicciones, metricas, rankings exploratorios y graficos.

Ejecucion en Kaggle: pegar en una celda o usar %run /kaggle/working/gat_heterogeneo_corregido.py
Ejecucion local:    python gat_heterogeneo_corregido.py
Prueba rapida:      GAT_SMOKE_TEST=1 python gat_heterogeneo_corregido.py

IMPORTANTE: Una red de correlaciones NO prueba interacciones ecologicas ni causalidad.
Ninguna precision esta garantizada. No optimizar en base al fold externo.
"""

import os
import json
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             f1_score, recall_score, confusion_matrix, log_loss,
                             roc_curve)

warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value encountered in divide')


@dataclass
class Config:
    seed: int = 42
    outer_folds: int = 5
    inner_validation_fraction: float = 0.20
    max_epochs: int = 250
    min_epochs: int = 30
    patience: int = 30
    min_delta: float = 0.0001
    hidden_channels: int = 32
    identity_dim: int = 16
    num_layers: int = 1       # Una capa = P->B, B->B y B->P; todas las fases afectan la salida
    dropout: float = 0.20
    learning_rate: float = 0.001
    weight_decay: float = 0.001
    grad_clip: float = 1.0
    class_weights: str = 'none'  # 'none' o 'balanced'; no introducir recargo adicional para HC
    top_k: int = 3
    corr_threshold: float = 0.35
    models: tuple = ('directo', 'hetero')  # Puede agregar 'bipartito'
    print_every: int = 25
    output_directory: str = '/kaggle/working/resultados_gat_heterogeneo' if Path('/kaggle/working').exists() else '/mnt/data/resultados_gat_heterogeneo'
    data_path: str = ''


CFG = Config()
if os.environ.get('GAT_SMOKE_TEST') == '1':
    CFG.outer_folds = 2
    CFG.max_epochs = 3
    CFG.min_epochs = 1
    CFG.patience = 2
    CFG.print_every = 1


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # El resultado puede variar ligeramente segun el dispositivo y version de CUDA.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def locate_dataset(path):
    paths = [path] if path else []
    paths += [
        '/kaggle/input/datasets/nico14h/metagenomica/dataset.csv',
        '/kaggle/working/dataset.csv',
        'dataset.csv',
        '/mnt/data/dataset.csv',
    ]
    for candidate in paths:
        if candidate and Path(candidate).is_file():
            return str(candidate)
    raise FileNotFoundError('No encontre dataset.csv. Configura CFG.data_path con la ruta correcta.')


def load_data(path):
    # Este archivo original usa tabulaciones; sep=None tambien funciona para CSV separado por comas.
    df = pd.read_csv(path, sep=None, engine='python')
    required = {'Bacteria_Limpia', 'Paciente', 'AbundanciaTaxonomica', 'Diagnosticos'}
    if not required.issubset(df.columns):
        raise ValueError(f'Columnas faltantes: {required - set(df.columns)}; disponibles: {df.columns.tolist()}')
    remove = {
        'Ralstonia', 'Burkholderia', 'Cupriavidus', 'Methylobacterium',
        'Bradyrhizobium', 'Rhizobium', 'Sphingomonas', 'Novosphingobium',
        'Acetobacter', 'Gluconobacter', 'Oenococcus', 'Cutibacterium',
        'uncultured', 'Desconocida'
    }
    df = df.loc[~df.Bacteria_Limpia.isin(remove)].copy()
    df = df.dropna(subset=['Bacteria_Limpia', 'Paciente', 'AbundanciaTaxonomica', 'Diagnosticos'])
    if df.duplicated(['Paciente', 'Bacteria_Limpia']).any():
        raise ValueError('Hay duplicados paciente-bacteria: revisar antes de construir la matriz.')
    if (df.groupby('Paciente')['Diagnosticos'].nunique() > 1).any():
        raise ValueError('Al menos un paciente tiene mas de un diagnostico.')
    df['Diagnosticos'] = pd.to_numeric(df['Diagnosticos'], errors='raise').astype(int)
    if not set(df.Diagnosticos.unique()).issubset({0, 1}):
        raise ValueError('Este script requiere Diagnosticos 0=HC y 1=TEA.')
    ids_pacientes = df['Paciente'].drop_duplicates().tolist()
    nombres_bacterias = df['Bacteria_Limpia'].drop_duplicates().tolist()
    pivot = df.pivot(index='Paciente', columns='Bacteria_Limpia', values='AbundanciaTaxonomica')
    raw = pivot.reindex(index=ids_pacientes, columns=nombres_bacterias).to_numpy(dtype=np.float32)
    etiquetas = df.groupby('Paciente')['Diagnosticos'].first().reindex(ids_pacientes).to_numpy(dtype=np.int64)
    present = np.isfinite(raw).astype(np.float32)
    return ids_pacientes, nombres_bacterias, raw, present, etiquetas


def sparsify_topk(corr, k=3, threshold=0.35):
    """A_ij=A_ji (top-k por union de vecinos), sin autoloops (se agregan en la atencion)."""
    n = corr.shape[0]
    c = np.nan_to_num(corr.copy(), nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(c, 0.0)
    mask = np.zeros((n, n), dtype=bool)
    for i in range(n):
        candidates = np.flatnonzero(np.abs(c[i]) >= threshold)
        if candidates.size:
            neighbors = candidates[np.argsort(-np.abs(c[i, candidates]))[:k]]
            mask[i, neighbors] = True
    mask |= mask.T
    adj = np.where(mask, c, 0.0).astype(np.float32)
    assert np.allclose(adj, adj.T, atol=1e-5), 'La red B-B debe ser simetrica'
    return adj


def corr_class(x):
    # x esta imputado con un piso calculado unicamente sobre entrenamiento.
    # Pearson sobre rCLR imputado NO es evidencia de simbiosis/competencia.
    with np.errstate(invalid='ignore', divide='ignore'):
        corr = np.corrcoef(x, rowvar=False)
    return np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)


def safe_standardize(x_all, train_idx):
    mean = x_all[train_idx].mean(axis=0, keepdims=True)
    std = x_all[train_idx].std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return np.clip((x_all - mean) / std, -6, 6).astype(np.float32)


def build_fold_graph(raw, present, y, train_idx, cfg, device):
    """Solo train_idx determina escalas, redes y topologia ecologica."""
    min_train = float(np.nanmin(raw[train_idx]))
    floor = min_train - 0.5
    filled = np.where(np.isfinite(raw), raw, floor).astype(np.float32)
    x_scaled = safe_standardize(filled, train_idx)
    m_train = present[train_idx]
    x_train = filled[train_idx]
    tea = x_train[y[train_idx] == 1]
    hc = x_train[y[train_idx] == 0]
    adj_red = sparsify_topk(corr_class(tea), cfg.top_k, cfg.corr_threshold)
    adj_green = sparsify_topk(corr_class(hc), cfg.top_k, cfg.corr_threshold)

    n_b = raw.shape[1]
    feature_bacteria = np.stack([
        (adj_red != 0).mean(axis=0),
        (adj_green != 0).mean(axis=0),
        np.maximum(adj_red, 0).mean(axis=0),
        np.maximum(-adj_red, 0).mean(axis=0),
        np.maximum(adj_green, 0).mean(axis=0),
        np.maximum(-adj_green, 0).mean(axis=0),
        np.linalg.norm(adj_red - adj_green, axis=1) / np.sqrt(n_b),
        m_train.mean(axis=0),
    ], axis=1).astype(np.float32)
    # Estandarizar a lo largo de bacterias no usa etiquetas ni pacientes del test.
    mu = feature_bacteria.mean(axis=0, keepdims=True)
    sd = feature_bacteria.std(axis=0, keepdims=True)
    feature_bacteria = (feature_bacteria - mu) / np.maximum(sd, 1e-6)
    feature_bacteria = np.clip(feature_bacteria, -6, 6)

    range_train = float(np.nanmax(raw[train_idx]) - floor)
    range_train = max(range_train, 1e-6)
    nonnegative_abundance = np.clip((filled - floor) / range_train, 0, 1) * present
    z_present = x_scaled * present
    # P<->B: atributos [abundancia positiva normalizada, z-score si presente, presencia].
    edge = np.stack([nonnegative_abundance, z_present, present], axis=-1).astype(np.float32)
    # Acceso directo a toda la firma microbiana, incluidas ausencias.
    x_direct = np.concatenate([x_scaled, present], axis=1).astype(np.float32)

    t = lambda a: torch.from_numpy(np.ascontiguousarray(a)).to(device)
    data = {
        'bact_features': t(feature_bacteria),
        'adj_red': t(adj_red),
        'adj_green': t(adj_green),
        'edges': t(edge),
        'x_direct': t(x_direct),
        'red_edges': int(np.count_nonzero(adj_red) // 2),
        'green_edges': int(np.count_nonzero(adj_green) // 2),
        'both_edges': int(np.count_nonzero((adj_red != 0) & (adj_green != 0)) // 2),
        'purple_train': int(present[train_idx].sum()),
        'floor': floor,
    }
    return data


def masked_softmax(scores, mask, dim):
    """Evita NaN si algun taxon no esta presente en ningun paciente de train."""
    a = F.softmax(scores.masked_fill(~mask, -1e9), dim=dim)
    a = a * mask.to(a.dtype)
    return a / a.sum(dim=dim, keepdim=True).clamp_min(1e-12)




## 2. Arquitectura de los tres tipos de relación

Las bacterias empiezan con su **identidad entrenable** más las **8 características ecológicas**. Los pacientes tienen representación por abundancia/presencia. En cada capa, las actualizaciones ocurren como P→B, B↔B y B→P. La rama directa clasifica sin pasar por la red; la compuerta aprendible regula la contribución ecológica.

La simetría se refiere a la **conectividad** del grafo; en atención GAT, el peso normalizado puede diferir entre extremos.

In [2]:
class SymmetricBacteriaAttention(nn.Module):
    """Atencion por tipo de red, soporte simetrico y atributo firmado [abs(r), r].

    La adyacencia es no dirigida (A_ij=A_ji), aunque como en GAT estandar
    los pesos de atencion por receptor pueden diferir al normalizar por fila.
    """
    def __init__(self, hidden, dropout):
        super().__init__()
        self.node_proj = nn.Linear(hidden, hidden, bias=False)
        self.edge_proj = nn.Linear(2, hidden, bias=False)
        self.score = nn.Linear(hidden, 1, bias=False)
        self.dropout = dropout

    def forward(self, h, adj):
        n = adj.shape[0]
        eye = torch.eye(n, device=adj.device, dtype=adj.dtype)
        signed = adj + eye
        mask = signed != 0
        edge_features = torch.stack((signed.abs(), signed), dim=-1)
        node = self.node_proj(h)
        pair = node[:, None, :] + node[None, :, :] + self.edge_proj(edge_features)
        scores = self.score(F.leaky_relu(pair, 0.2)).squeeze(-1)
        alpha = masked_softmax(scores, mask, dim=1)
        alpha_drop = F.dropout(alpha, p=self.dropout, training=self.training)
        # Sin multiplicar la representacion por correlaciones negativas.
        return F.elu(alpha_drop @ node)


class HeteroMicrobiomeGAT(nn.Module):
    """Grafo heterogeneo no dirigido; dos tipos de B-B, un tipo de P-B.

    El intercambio de mensajes P->B solo incorpora pacientes de TRAIN.
    La rama directa evita perder señales fuertes de abundancia/presencia.
    """
    def __init__(self, num_bacteria, cfg, mode):
        super().__init__()
        if mode not in ('directo', 'bipartito', 'hetero'):
            raise ValueError('Modo desconocido')
        self.mode = mode
        self.cfg = cfg
        h = cfg.hidden_channels
        self.direct_classifier = nn.Linear(num_bacteria * 2, 2)
        if mode == 'directo':
            return
        self.embedding_bacteria = nn.Embedding(num_bacteria, cfg.identity_dim)
        self.bacteria_initial = nn.Sequential(nn.Linear(8 + cfg.identity_dim, h), nn.LayerNorm(h), nn.ELU())
        self.patient_initial = nn.Sequential(nn.Linear(num_bacteria * 2, h), nn.LayerNorm(h), nn.ELU())
        self.ecological_gate = nn.Parameter(torch.tensor(-2.0))
        self.feedback_gate = nn.Parameter(torch.tensor(-2.0))
        self.bb_gate = nn.Parameter(torch.tensor(-2.0))
        self.layers = nn.ModuleList()
        for _ in range(cfg.num_layers):
            # Las relaciones P<->B admiten propagacion en ambos sentidos
            # con transformaciones distintas por tipo de nodo.
            block = nn.ModuleDict({
                'p_to_b_patient': nn.Linear(h, h, bias=False),
                'p_to_b_bacteria': nn.Linear(h, h, bias=False),
                'p_to_b_edge': nn.Linear(3, h, bias=False),
                'p_to_b_attention': nn.Linear(h, 1, bias=False),
                'p_to_b_message': nn.Linear(h + 3, h),
                'norm_b1': nn.LayerNorm(h),
                'red': SymmetricBacteriaAttention(h, cfg.dropout),
                'green': SymmetricBacteriaAttention(h, cfg.dropout),
                'fuse_bb': nn.Sequential(nn.Linear(h * 4, h), nn.LayerNorm(h), nn.ELU()),
                'norm_b2': nn.LayerNorm(h),
                'b_to_p_patient': nn.Linear(h, h, bias=False),
                'b_to_p_bacteria': nn.Linear(h, h, bias=False),
                'b_to_p_edge': nn.Linear(3, h, bias=False),
                'b_to_p_attention': nn.Linear(h, 1, bias=False),
                'b_to_p_message': nn.Linear(h + 3, h),
                'norm_p': nn.LayerNorm(h),
            })
            self.layers.append(block)
        self.eco_classifier = nn.Sequential(
            nn.LayerNorm(h), nn.Dropout(cfg.dropout),
            nn.Linear(h, max(8, h // 2)), nn.ELU(),
            nn.Dropout(cfg.dropout), nn.Linear(max(8, h // 2), 2),
        )

    def forward(self, graph, train_mask):
        direct = self.direct_classifier(graph['x_direct'])
        if self.mode == 'directo':
            return direct, {'eco_gate': None, 'alpha_b2p': None}
        edge = graph['edges']
        mask_pb = edge[..., 2] > 0
        p, b, _ = edge.shape
        bact_ids = torch.arange(b, device=edge.device)
        h_b = self.bacteria_initial(torch.cat([
            graph['bact_features'], self.embedding_bacteria(bact_ids)
        ], dim=1))
        h_p = self.patient_initial(graph['x_direct'])
        last_alpha = None
        for block in self.layers:
            # PASO 1: pacientes de TRAIN -> bacterias (P<->B).
            # Este paso ocurre ANTES de la clasificacion, incluso con una sola capa.
            p_proj = block['p_to_b_patient'](h_p)
            b_proj = block['p_to_b_bacteria'](h_b)
            e_proj = block['p_to_b_edge'](edge)
            p2b_energy = F.leaky_relu(p_proj[:, None, :] + b_proj[None, :, :] + e_proj, 0.2)
            p2b_scores = block['p_to_b_attention'](p2b_energy).squeeze(-1)
            p2b_mask = mask_pb & train_mask[:, None]
            p2b_alpha = masked_softmax(p2b_scores, p2b_mask, dim=0)
            p_msg = p_proj[:, None, :] * edge[..., 0:1]
            p2b_input = torch.cat([p_msg.expand(p, b, -1), edge], dim=-1)
            p2b_messages = F.elu(block['p_to_b_message'](p2b_input))
            aggregate_b = (p2b_alpha[..., None] * p2b_messages).sum(dim=0)
            h_b = block['norm_b1'](h_b + torch.sigmoid(self.feedback_gate) *
                                     F.dropout(aggregate_b, p=self.cfg.dropout, training=self.training))
            # PASO 2: bacteria <-> bacteria; diferenciamos redes TEA y HC.
            if self.mode == 'hetero':
                red = block['red'](h_b, graph['adj_red'])
                green = block['green'](h_b, graph['adj_green'])
                fusion = block['fuse_bb'](torch.cat([h_b, red, green, red - green], dim=-1))
                h_b = block['norm_b2'](h_b + torch.sigmoid(self.bb_gate) *
                                         F.dropout(fusion, p=self.cfg.dropout, training=self.training))
            # PASO 3: bacterias -> pacientes (P<->B).
            p_proj = block['b_to_p_patient'](h_p)
            b_proj = block['b_to_p_bacteria'](h_b)
            e_proj = block['b_to_p_edge'](edge)
            energy = F.leaky_relu(p_proj[:, None, :] + b_proj[None, :, :] + e_proj, 0.2)
            scores = block['b_to_p_attention'](energy).squeeze(-1)
            alpha = masked_softmax(scores, mask_pb, dim=1)
            b_msg = b_proj[None, :, :] * edge[..., 0:1]
            b2p_input = torch.cat([b_msg.expand(p, b, -1), edge], dim=-1)
            b2p_messages = F.elu(block['b_to_p_message'](b2p_input))
            aggregate_p = (alpha[..., None] * b2p_messages).sum(dim=1)
            h_p = block['norm_p'](h_p + F.dropout(aggregate_p, p=self.cfg.dropout, training=self.training))
            last_alpha = alpha
        logits = direct + torch.sigmoid(self.ecological_gate) * self.eco_classifier(h_p)
        return logits, {'eco_gate': torch.sigmoid(self.ecological_gate), 'alpha_b2p': last_alpha}




## 3. Entrenamiento y validación rigurosa

En cada fold externo, la validación interna selecciona el número de épocas. Luego se reconstruye el grafo usando solo todos los pacientes de entrenamiento externo, se entrena nuevamente desde cero y se evalúa una vez sobre los pacientes externos. No se trasladan niños desde entrenamiento a validación ni se reutilizan etiquetas externas para construir la red.

In [3]:
def get_weights(y_train, cfg, device):
    if cfg.class_weights == 'none':
        return None
    if cfg.class_weights == 'balanced':
        counts = np.bincount(y_train, minlength=2).astype(float)
        weights = len(y_train) / (2.0 * np.maximum(counts, 1))
        return torch.tensor(weights, dtype=torch.float32, device=device)
    raise ValueError('class_weights debe ser none o balanced')


def make_mask(train_idx, n, device):
    mask = torch.zeros(n, dtype=torch.bool, device=device)
    mask[torch.as_tensor(train_idx, dtype=torch.long, device=device)] = True
    return mask


def run_epochs(model, graph, y_torch, train_idx, train_mask, cfg,
               epochs, device, valid_idx=None, early_stop=False):
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate,
                                  weight_decay=cfg.weight_decay)
    weight = get_weights(y_torch[train_idx].cpu().numpy(), cfg, device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    best_loss, best_epoch, stale = float('inf'), 1, 0
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(graph, train_mask)
        loss = criterion(logits[train_idx], y_torch[train_idx])
        if not torch.isfinite(loss):
            raise ValueError(f'Loss no finita en epoca {epoch}; revisar normalizacion y LR.')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        if early_stop:
            model.eval()
            with torch.no_grad():
                val_logits, _ = model(graph, train_mask)
                # Evaluamos la misma CrossEntropy sin pesos por clases para comparar calibracion.
                val_loss = float(F.cross_entropy(val_logits[valid_idx], y_torch[valid_idx]).item())
            if val_loss < best_loss - cfg.min_delta:
                best_loss, best_epoch, stale = val_loss, epoch, 0
            else:
                stale += 1
            if epoch >= cfg.min_epochs and stale >= cfg.patience:
                break
    if not early_stop:
        best_epoch = epochs
    return best_epoch, best_loss


def evaluate_oof(y, probabilities):
    predictions = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(y, predictions),
        'balanced_accuracy': balanced_accuracy_score(y, predictions),
        'roc_auc': roc_auc_score(y, probabilities),
        'f1_tea': f1_score(y, predictions, zero_division=0),
        'sensitivity_tea': tp / max(1, tp + fn),
        'specificity_hc': tn / max(1, tn + fp),
        'log_loss': log_loss(y, probabilities, labels=[0, 1]),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
    }


def train_one_model(mode, outer_splits, raw, present, y, patient_ids, bacteria_names, cfg, device, out_dir):
    print(f'\n========= MODELO: {mode.upper()} =========')
    n, b = raw.shape
    y_torch = torch.tensor(y, dtype=torch.long, device=device)
    oof_probability = np.full(n, np.nan, dtype=float)
    fold_rows = []
    attention_tea = np.zeros(b, dtype=float)
    attention_hc = np.zeros(b, dtype=float)
    count_tea, count_hc = 0, 0
    direct_coef = np.zeros(2 * b, dtype=float)
    eco_gates = []

    for fold, (train_idx, test_idx) in enumerate(outer_splits, 1):
        # Separar INNER train/valid SOLO dentro del train externo.
        inner_train_idx, inner_val_idx = train_test_split(
            train_idx, test_size=cfg.inner_validation_fraction,
            stratify=y[train_idx], random_state=cfg.seed + fold,
        )
        # Los dos grafos INNER se construyen solo usando etiquetas del INNER train.
        graph_inner = build_fold_graph(raw, present, y, inner_train_idx, cfg, device)
        train_mask_inner = make_mask(inner_train_idx, n, device)
        set_seed(cfg.seed + fold * 100)
        model_inner = HeteroMicrobiomeGAT(b, cfg, mode).to(device)
        best_epoch, best_loss = run_epochs(
            model_inner, graph_inner, y_torch, inner_train_idx, train_mask_inner,
            cfg, cfg.max_epochs, device, valid_idx=inner_val_idx, early_stop=True)
        del model_inner, graph_inner
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        # Reentrenar DESDE CERO sobre todo el train externo, por best_epoch epocas.
        # La prueba externa no se uso para elegir best_epoch, topk, ni nada mas.
        graph_outer = build_fold_graph(raw, present, y, train_idx, cfg, device)
        train_mask_outer = make_mask(train_idx, n, device)
        set_seed(cfg.seed + fold * 100)
        model = HeteroMicrobiomeGAT(b, cfg, mode).to(device)
        run_epochs(model, graph_outer, y_torch, train_idx, train_mask_outer,
                   cfg, best_epoch, device, early_stop=False)
        model.eval()
        with torch.no_grad():
            logits, extras = model(graph_outer, train_mask_outer)
            probs = F.softmax(logits[test_idx], dim=1)[:, 1].cpu().numpy()
            oof_probability[test_idx] = probs
            eco_gate = float(extras['eco_gate'].item()) if extras['eco_gate'] is not None else np.nan
            if extras['alpha_b2p'] is not None:
                attentions = extras['alpha_b2p'][test_idx].cpu().numpy()
                ya = y[test_idx]
                attention_tea += attentions[ya == 1].sum(axis=0)
                attention_hc += attentions[ya == 0].sum(axis=0)
                count_tea += int((ya == 1).sum())
                count_hc += int((ya == 0).sum())
            w = model.direct_classifier.weight.detach().cpu().numpy()
            direct_coef += (w[1] - w[0]) / cfg.outer_folds
        metrics = evaluate_oof(y[test_idx], probs)
        fold_rows.append({'model': mode, 'fold': fold, 'train_count': len(train_idx),
                          'test_count': len(test_idx), 'tea_train': int(y[train_idx].sum()),
                          'hc_train': int((y[train_idx] == 0).sum()),
                          'best_epoch_inner': best_epoch,
                          'best_inner_val_loss': best_loss,
                          'red_edges': graph_outer['red_edges'],
                          'green_edges': graph_outer['green_edges'],
                          'common_edges': graph_outer['both_edges'],
                          'purple_train': graph_outer['purple_train'],
                          'eco_gate': eco_gate, **metrics})
        eco_gates.append(eco_gate)
        print(f"Fold {fold}: train={len(train_idx)} test={len(test_idx)} "
              f"best_epoch={best_epoch}, TEA-BB={graph_outer['red_edges']}, "
              f"HC-BB={graph_outer['green_edges']}  "
              f"ACC={metrics['accuracy']:.3f} AUC={metrics['roc_auc']:.3f} "
              f"Sens={metrics['sensitivity_tea']:.3f} Spec={metrics['specificity_hc']:.3f}")
        del model, graph_outer
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    if np.isnan(oof_probability).any():
        raise RuntimeError('OOF incompletas: algunos pacientes no fueron evaluados.')
    global_metrics = evaluate_oof(y, oof_probability)
    df_folds = pd.DataFrame(fold_rows)
    df_folds.to_csv(out_dir / f'metricas_por_fold_{mode}.csv', index=False)
    preds = pd.DataFrame({'Paciente': patient_ids, 'Real': y,
                          'Probabilidad_TEA': oof_probability,
                          'Prediccion_TEA': (oof_probability >= 0.5).astype(int)})
    preds.to_csv(out_dir / f'predicciones_oof_{mode}.csv', index=False)
    ranks = pd.DataFrame({
        'Bacteria': bacteria_names,
        'Coef_rCLR_directo_aprox': direct_coef[:b],
        'Coef_presencia_directo_aprox': direct_coef[b:],
        'Atencion_media_TEA_OOF': attention_tea / max(count_tea, 1),
        'Atencion_media_HC_OOF': attention_hc / max(count_hc, 1),
    })
    ranks['Importancia_lineal_absoluta'] = (ranks.Coef_rCLR_directo_aprox.abs() +
                                           ranks.Coef_presencia_directo_aprox.abs())
    ranks.sort_values('Importancia_lineal_absoluta', ascending=False).to_csv(
        out_dir / f'ranking_exploratorio_{mode}.csv', index=False)
    print('OOF GLOBAL:', ', '.join(f'{k}={v:.4f}' for k, v in global_metrics.items() if k not in ('tn', 'fp', 'fn', 'tp')))
    return {'model': mode, **global_metrics}, (y, oof_probability)


def plot_oof_curves(all_predictions, out_dir):
    plt.figure(figsize=(7, 5))
    for mode, (y, probs) in all_predictions.items():
        fpr, tpr, _ = roc_curve(y, probs)
        auc = roc_auc_score(y, probs)
        plt.plot(fpr, tpr, label=f'{mode} (AUC={auc:.3f})')
    plt.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
    plt.xlabel('1 - Especificidad (FPR)')
    plt.ylabel('Sensibilidad TEA (TPR)')
    plt.title('Curvas ROC - Predicciones externas OOF')
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / 'curvas_roc_oof.png', dpi=180)
    plt.close()


def main():
    set_seed(CFG.seed)
    # Limita overhead de CPU en notebooks para los tensores pequenos de este dataset.
    if not torch.cuda.is_available():
        torch.set_num_threads(min(4, os.cpu_count() or 1))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    out_dir = Path(CFG.output_directory)
    out_dir.mkdir(parents=True, exist_ok=True)
    dataset_path = locate_dataset(CFG.data_path)
    patient_ids, bacteria_names, raw, present, y = load_data(dataset_path)
    print(f'Dataset: {len(patient_ids)} pacientes, {len(bacteria_names)} bacterias, '
          f'{int((y==1).sum())} TEA, {int((y==0).sum())} HC, '
          f'{int(present.sum())} aristas P<->B (sin contar inversas)')
    print(f'Device={device} | dataset={dataset_path} | resultados={out_dir}')
    print('IMPORTANTE: Grafos rojo/verde construidos exclusivamente con TRAIN de cada split.')
    with open(out_dir / 'configuracion.json', 'w', encoding='utf-8') as fp:
        json.dump(asdict(CFG), fp, indent=2, ensure_ascii=False)
    outer_cv = StratifiedKFold(n_splits=CFG.outer_folds, shuffle=True, random_state=CFG.seed)
    outer_splits = list(outer_cv.split(raw, y))
    # Chequeo: cada paciente aparece exactamente una vez en test externo.
    counts = np.bincount(np.concatenate([idx_test for _, idx_test in outer_splits]), minlength=len(y))
    assert np.all(counts == 1), 'Cada paciente debe aparecer exactamente una vez en test externo'
    global_rows = []
    all_predictions = {}
    for mode in CFG.models:
        metrics, preds = train_one_model(mode, outer_splits, raw, present, y,
                                         patient_ids, bacteria_names, CFG, device, out_dir)
        global_rows.append(metrics)
        all_predictions[mode] = preds
    result = pd.DataFrame(global_rows).set_index('model')
    result.to_csv(out_dir / 'comparacion_modelos.csv')
    plot_oof_curves(all_predictions, out_dir)
    print('\n=== COMPARACION GLOBAL OOF (5 folds si no es smoke test) ===')
    print(result[['accuracy','balanced_accuracy','roc_auc','f1_tea',
                  'sensitivity_tea','specificity_hc']].round(4).to_string())
    print('\nArchivos guardados en:', out_dir)
    print('Los rankings son exploratorios, NO biomarcadores clinicamente validados.')
    return result




## 4. Ejecutar y guardar los resultados

Se generan `comparacion_modelos.csv`, `metricas_por_fold_*.csv`, `predicciones_oof_*.csv`, `ranking_exploratorio_*.csv` y `curvas_roc_oof.png`. Por defecto compara un modelo directo y otro heterogéneo. Podés agregar `"bipartito"` a `CFG.models`.

La tabla de rankings es exploratoria: ni las correlaciones ni la atención de GAT demuestran competencia/cooperación ecológica ni biomarcadores clínicamente validados.

In [4]:
resultados = main()
resultados

Dataset: 121 pacientes, 152 bacterias, 71 TEA, 50 HC, 10679 aristas P<->B (sin contar inversas)
Device=cuda | dataset=/kaggle/input/datasets/nico14h/metagenomica/dataset.csv | resultados=/kaggle/working/resultados_gat_heterogeneo
IMPORTANTE: Grafos rojo/verde construidos exclusivamente con TRAIN de cada split.

========= MODELO: DIRECTO =========
Fold 1: train=96 test=25 best_epoch=250, TEA-BB=325, HC-BB=316  ACC=0.880 AUC=0.947 Sens=0.933 Spec=0.800
Fold 2: train=97 test=24 best_epoch=40, TEA-BB=330, HC-BB=320  ACC=0.917 AUC=0.957 Sens=0.929 Spec=0.900
Fold 3: train=97 test=24 best_epoch=250, TEA-BB=323, HC-BB=328  ACC=0.917 AUC=0.971 Sens=1.000 Spec=0.800
Fold 4: train=97 test=24 best_epoch=241, TEA-BB=320, HC-BB=337  ACC=0.917 AUC=0.957 Sens=0.929 Spec=0.900
Fold 5: train=97 test=24 best_epoch=250, TEA-BB=314, HC-BB=337  ACC=0.833 AUC=0.929 Sens=0.857 Spec=0.800
OOF GLOBAL: accuracy=0.8926, balanced_accuracy=0.8848, roc_auc=0.9515, f1_tea=0.9103, sensitivity_tea=0.9296, specificity_

,accuracy,balanced_accuracy,roc_auc,f1_tea,sensitivity_tea,specificity_hc,log_loss,tn,fp,fn,tp
model,,,,,,,,,,,
directo,0.892562,0.884789,0.951549,0.910345,0.929577,0.84,0.274214,42,8,5,66
hetero,0.859504,0.847746,0.943662,0.884354,0.915493,0.78,0.313907,39,11,6,65
